## Async Random Sentences to S3

Create a batch of random sentences, store them in S3 through LAILA, persist a `{nickname: global_id}` manifest, and await the batched `GroupFuture` directly.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import random

import laila
from laila.pool import S3Pool


In [ ]:
POOL_NICKNAME = "my_sentences_pool"
MANIFEST_NICKNAME = "my_sentences_manifest"
NUM_SENTENCES = 25

subjects = ["The robot", "A scientist", "My neighbor", "The artist", "An astronaut", "The teacher"]
verbs = ["builds", "discovers", "writes", "studies", "launches", "observes"]
objects = ["a tiny engine", "a strange pattern", "a new theorem", "the old telescope", "a paper airplane", "a hidden garden"]
adverbs = ["carefully", "enthusiastically", "quietly", "overnight", "without hesitation", "before sunrise"]

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
        nickname=POOL_NICKNAME,
    )


def make_random_sentence() -> str:
    return " ".join(
        [
            random.choice(subjects),
            random.choice(verbs),
            random.choice(objects),
            random.choice(adverbs) + ".",
        ]
    )


In [ ]:
s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)

sentence_entries = []
sentence_manifest = {}
for i in range(NUM_SENTENCES):
    sentence_name = f"random_sentence_{i}"
    sentence = make_random_sentence()
    entry = laila.constant(data=sentence, nickname=sentence_name)
    sentence_entries.append(entry)
    sentence_manifest[sentence_name] = entry.global_id

manifest_entry = laila.constant(data=sentence_manifest, nickname=MANIFEST_NICKNAME)

list(sentence_manifest.items())[:3]

In [ ]:
upload_futures = laila.memorize(sentence_entries, pool_nickname=POOL_NICKNAME)
manifest_future = laila.memorize(manifest_entry, pool_nickname=POOL_NICKNAME)

print("Initial sentence upload status:", upload_futures.status)
await upload_futures
await manifest_future

print("\nFinal sentence upload status:", upload_futures.status)
print("Uploaded sentence futures:", len(upload_futures))
print("Manifest global id:", manifest_entry.global_id)

In [ ]:
manifest_future = laila.remember(manifest_entry.global_id, pool_nickname=POOL_NICKNAME)
remembered_manifest_entry = await manifest_future
remembered_manifest = dict(remembered_manifest_entry.data)

remember_names = list(remembered_manifest.keys())
remember_futures = laila.remember(
    list(remembered_manifest.values()),
    pool_nickname=POOL_NICKNAME,
)

remembered_entries = await remember_futures
remembered_sentences = {
    name: entry.data
    for name, entry in zip(remember_names, remembered_entries)
}

list(remembered_sentences.items())[:3]